In [1]:
# Import libraries for modeling, text processing, and validation

import pandas as pd
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier  # Optimized Gradient Boosting classifier

# Load the '20 Newsgroups' dataset, a collection of English texts classified by topic

data = fetch_20newsgroups(subset='all', remove=())  # The full dataset is loaded without removing any part

# Create a DataFrame with the document text and its numeric label
df = pd.DataFrame({
    'text': data.data,
    'original_label': data.target
})

# Add a column with the name of the textual category corresponding to each label
category_names = data.target_names
df['topic'] = df['original_label'].apply(lambda x: category_names[x])

# Definition of a binary variable "sport"
# Positive classes: categories related to sports and motor vehicles

sport_categories = ['rec.sport.baseball', 'rec.sport.hockey', 'rec.autos', 'rec.motorcycles']
df['sport'] = df['topic'].apply(lambda x: 1 if x in sport_categories else 0)

# Split the data into features (X) and binary labels (y)

X = df['text'].values
y = df['sport'].values

# Definition of a custom stopword list
# These words will be ignored during vectorization

stopwords = [
    'as', 'an', 'the', 'in', 'on', 'at', 'to', 'of', 'and', 'or',
    'is', 'it', 'for', 'with', 'that', 'this', 'was', 'be',
    'are', 'were', 'been', 'from', 'by', 'about', 'into', 'out',
    'up', 'down', 'over', 'under', 'then', 'than', 'so', 'but', 'not'
]

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Parameters to explore: number of topics (k) for LDA

k_values = [20]  # In this example, only k = 20 is tested

# Initialize the list to store the final results
results = []

In [2]:
i=0
# Cross-validation configuration (5 splits)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Main loop: for each value of k (topics), cross-validation is performed
# with XGBoost trained on the LDA thematic representations


for k in k_values:
    scores = []  # List to store the AUCs of each fold

    for train_idx, test_idx in cv.split(X, y):
        # Split into training and test sets
        X_train_raw, X_test_raw = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Vectorize the texts with CountVectorizer
        # It is limited to the 10,000 most frequent words and stopwords are filtered out

        vectorizer = CountVectorizer(max_features=10000, stop_words=stopwords)
        X_train_counts = vectorizer.fit_transform(X_train_raw)
        X_test_counts = vectorizer.transform(X_test_raw)

        # Dimensionality reduction with LDA (Latent Dirichlet Allocation)
        # Each document is represented as a probability distribution over k topics

        lda = LatentDirichletAllocation(n_components=k, random_state=42)
        X_train_topics = lda.fit_transform(X_train_counts)
        X_test_topics = lda.transform(X_test_counts)


        # Train the XGBoost model (Gradient Boosting for binary classification)
        # Common parameters: maximum depth, number of trees, metric, and seed

        model = XGBClassifier(
            max_depth=6,
            n_estimators=500,
            use_label_encoder=False,  # Disabled to avoid old warnings
            eval_metric='logloss',
            random_state=42
        )
        i=i+1
        model.fit(X_train_topics, y_train)
        print("Iteration: "+str(i) )
        # Probability prediction and AUC-ROC metric computation
        # This metric evaluates the quality of probabilistic classification

        probabilities = model.predict_proba(X_test_topics)[:, 1]
        score = roc_auc_score(y_test, probabilities)
        scores.append(score)
        print("AUC-ROC: Fold " + str(i) +  ":" + str(score))  # Print the AUC score of the current fold

    # Store the mean AUC-ROC value for this k value

    results.append({
        'k': k,
        'mean_auc_roc': np.mean(scores)
    })


Iteration: 1
AUC ROC: Fold 1:0.9640405288040741


Iteration: 2
AUC ROC: Fold 2:0.9694402047235843


Iteration: 3
AUC ROC: Fold 3:0.98127959001195


Iteration: 4
AUC ROC: Fold 4:0.9795027103225513


Iteration: 5
AUC ROC: Fold 5:0.9772453506913165


In [3]:
# Organize the results
df_results = pd.DataFrame(results)

# Display the results table
print("\n Cross-validation results (mean AUC-ROC for k=20):\n")
print(df_results.to_string(index=False, float_format="%.4f"))



 Cross-validation results (mean AUC-ROC for k=20):

 k  mean_auc_roc
20        0.9743
